In [1]:
"""
Wolfpack Trend Strategy — Local Python Simulation
===================================================
Replicates the QC backtest logic using synthetic but realistic price data
for the 30 DJIA stocks over 2020-01-01 to 2022-01-01.

Outputs (saved to ./output/):
  daily_snapshots.csv
  positions.csv
  signals.csv
  slippage.csv
  targets.csv
  order_events.csv
"""

import os
import math
import numpy as np
import pandas as pd
from datetime import date, timedelta

# ── Config ────────────────────────────────────────────────────────────────────
EQUITY_UNIVERSE = [
    "AAPL","AMGN","AXP","BA","CAT","CRM","CSCO","CVX","DIS","DOW",
    "GS","HD","HON","IBM","INTC","JNJ","JPM","KO","MCD","MMM",
    "MRK","MSFT","NKE","PG","TRV","UNH","V","VZ","WBA","WMT",
]
START_DATE   = date(2020, 1, 1)
END_DATE     = date(2022, 1, 1)
WARMUP_DAYS  = 252
STARTING_CASH = 100_000.0

# Alpha params
SHORT_PERIOD  = 20
MEDIUM_PERIOD = 63
LONG_PERIOD   = 252
ATR_PERIOD    = 14
SIGNAL_WEIGHTS      = (0.2, 0.5, 0.3)
SIGNAL_TEMPERATURE  = 3.0
ALPHA_MIN_MAGNITUDE = 0.05
REBALANCE_INTERVAL  = 5  # trading days

# Portfolio params
TARGET_VOL   = 0.10
MAX_GROSS    = 1.50
MAX_NET      = 0.50
MAX_WEIGHT   = 0.10
VOL_LOOKBACK = 63
SCALING_DAYS = 5
DEAD_BAND    = 0.015

# Execution params
STRONG_THRESHOLD   = 0.70
MODERATE_THRESHOLD = 0.30
MODERATE_OFFSET    = 0.005
WEAK_OFFSET        = 0.015

OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)

# ── Synthetic price generation ────────────────────────────────────────────────
def generate_prices(tickers, start, end, warmup=252):
    """Generate correlated GBM prices with regime shifts (COVID crash + recovery)."""
    total_start = start - timedelta(days=warmup * 1.5)
    biz_dates = pd.bdate_range(total_start, end)

    n = len(biz_dates)
    nt = len(tickers)

    # Base params per stock (annualised)
    np.random.seed(42)
    ann_ret  = np.random.uniform(0.05, 0.25, nt)
    ann_vol  = np.random.uniform(0.18, 0.45, nt)

    # Correlation matrix (moderate market correlation ~0.5)
    corr = np.full((nt, nt), 0.50)
    np.fill_diagonal(corr, 1.0)
    L = np.linalg.cholesky(corr)

    dt = 1 / 252
    daily_ret  = ann_ret  * dt
    daily_vol  = ann_vol  * np.sqrt(dt)

    # Simulate
    z = np.random.randn(n, nt)
    z_corr = z @ L.T

    log_returns = daily_ret - 0.5 * daily_vol**2 + daily_vol * z_corr

    # COVID crash: Feb 20 – Mar 23 2020
    for i, d in enumerate(biz_dates):
        if date(2020, 2, 20) <= d.date() <= date(2020, 3, 23):
            log_returns[i] -= 0.025 + np.random.uniform(0, 0.01, nt)
        elif date(2020, 3, 24) <= d.date() <= date(2020, 8, 31):
            log_returns[i] += 0.008 + np.random.uniform(0, 0.005, nt)

    prices = np.exp(np.cumsum(log_returns, axis=0))
    # Normalise to start at realistic prices
    start_prices = np.random.uniform(50, 400, nt)
    prices = prices / prices[0] * start_prices

    df = pd.DataFrame(prices, index=biz_dates, columns=tickers)
    return df

print("Generating synthetic price data...")
all_prices = generate_prices(EQUITY_UNIVERSE, START_DATE, END_DATE, WARMUP_DAYS)
spy_prices = all_prices.mean(axis=1)  # equal-weight index as SPY proxy

# Split warmup vs backtest
bt_dates = pd.bdate_range(START_DATE, END_DATE - timedelta(days=1))
warmup_prices = all_prices[all_prices.index < pd.Timestamp(START_DATE)]
bt_prices     = all_prices[all_prices.index.isin(bt_dates)]
print(f"Price data: {len(all_prices)} total days | {len(bt_prices)} backtest days")

# ── Indicator helpers ─────────────────────────────────────────────────────────
def sma(series, period):
    return series.rolling(period).mean()

def atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# Pre-compute indicators on full price history
print("Computing indicators...")
# Use all_prices for indicator warmup
highs  = all_prices * (1 + np.random.uniform(0, 0.01, all_prices.shape))
lows   = all_prices * (1 - np.random.uniform(0, 0.01, all_prices.shape))
closes = all_prices.copy()

sma_short  = closes.apply(lambda c: sma(c, SHORT_PERIOD))
sma_medium = closes.apply(lambda c: sma(c, MEDIUM_PERIOD))
sma_long   = closes.apply(lambda c: sma(c, LONG_PERIOD))
atr_vals   = pd.DataFrame({t: atr(highs[t], lows[t], closes[t], ATR_PERIOD)
                            for t in EQUITY_UNIVERSE})

# ── Signal computation ────────────────────────────────────────────────────────
def compute_composite_signal(price, sma_s, sma_m, sma_l, atr_v,
                              weights=(0.2,0.5,0.3), temperature=3.0, min_mag=0.05):
    if any(pd.isna([sma_s, sma_m, sma_l, atr_v])) or atr_v < 1e-8:
        return None
    s_short  = (price - sma_s) / atr_v
    s_medium = (price - sma_m) / atr_v
    s_long   = (price - sma_l) / atr_v
    composite = weights[0]*s_short + weights[1]*s_medium + weights[2]*s_long
    mag = math.tanh(composite / temperature)
    if abs(mag) < min_mag:
        return None
    return mag

# ── Scaling schedule ──────────────────────────────────────────────────────────
def build_scaling_schedule(n_days, front_load_factor=1.0):
    raw = [front_load_factor ** i for i in range(n_days)]
    total = sum(raw)
    cumulative = 0.0
    schedule = []
    for v in raw:
        cumulative += v / total
        schedule.append(round(min(cumulative, 1.0), 6))
    return schedule

strong_sched   = build_scaling_schedule(SCALING_DAYS, 2.0)
moderate_sched = build_scaling_schedule(SCALING_DAYS, 1.3)
weak_sched     = build_scaling_schedule(SCALING_DAYS, 1.0)

def get_schedule(sig_str):
    if sig_str >= STRONG_THRESHOLD:   return strong_sched
    if sig_str >= MODERATE_THRESHOLD: return moderate_sched
    return weak_sched

# ── Portfolio construction helpers ────────────────────────────────────────────
def estimate_vol(weights, returns_dict, min_obs=20):
    syms = [s for s in weights if s in returns_dict and len(returns_dict[s]) >= min_obs]
    if not syms:
        return None
    w = np.array([weights[s] for s in syms])
    rets = np.array([returns_dict[s][-VOL_LOOKBACK:] for s in syms])
    cov = np.cov(rets) * 252
    if cov.ndim == 0:
        cov = np.array([[cov]])
    return float(np.sqrt(w @ cov @ w))

def apply_per_name_cap(weights, cap):
    return {s: np.clip(w, -cap, cap) for s, w in weights.items()}

def apply_gross_cap(weights, max_gross):
    gross = sum(abs(w) for w in weights.values())
    if gross > max_gross:
        scale = max_gross / gross
        weights = {s: w * scale for s, w in weights.items()}
    return weights

def apply_net_cap(weights, max_net):
    net = sum(weights.values())
    if abs(net) > max_net:
        excess = net - np.sign(net) * max_net
        n = len(weights)
        weights = {s: w - excess/n for s, w in weights.items()}
    return weights

# ── Backtest state ────────────────────────────────────────────────────────────
cash        = STARTING_CASH
positions   = {t: 0.0 for t in EQUITY_UNIVERSE}   # shares held
nav_history = []

# Rolling returns
rolling_returns = {t: [] for t in EQUITY_UNIVERSE}
prev_close      = {}

# PCM state
weekly_targets       = {}
signal_strengths     = {}
symbol_scale_state   = {}
previous_weekly_targets = {}
last_classifications = {}
current_week_id      = None
is_rebalance_day     = False
trading_days_since_rebalance = None

# Alpha state
cached_signals = {}
alpha_trading_days = None
last_emit_date = None

# Order tracking
order_id_counter = 0
open_limit_orders = []   # list of dicts
order_week_ids    = {}

# Logs
log_snapshots   = []
log_positions   = []
log_signals     = []
log_slippage    = []
log_targets     = []
log_order_events = []

def next_order_id():
    global order_id_counter
    order_id_counter += 1
    return order_id_counter

def get_nav(date_idx, price_row):
    port_val = sum(positions[t] * price_row[t] for t in EQUITY_UNIVERSE)
    return cash + port_val

def get_tier_label(sig_str):
    if sig_str >= STRONG_THRESHOLD:   return "strong"
    if sig_str >= MODERATE_THRESHOLD: return "moderate"
    return "weak"

print("Running backtest simulation...")

prev_nav = None
starting_cash = STARTING_CASH

for day_idx, dt in enumerate(bt_dates):
    price_row = bt_prices.loc[dt]

    # ── Update rolling returns ────────────────────────────────────────────────
    for t in EQUITY_UNIVERSE:
        p = price_row[t]
        if t in prev_close and prev_close[t] > 0:
            r = p / prev_close[t] - 1.0
            rolling_returns[t].append(r)
            if len(rolling_returns[t]) > VOL_LOOKBACK:
                rolling_returns[t].pop(0)
        prev_close[t] = p

    nav = get_nav(day_idx, price_row)

    # ── Alpha: compute signals every REBALANCE_INTERVAL days ─────────────────
    if alpha_trading_days is None or alpha_trading_days >= REBALANCE_INTERVAL:
        alpha_trading_days = 1
        is_rebalance_day   = True
        cached_signals     = {}

        for t in EQUITY_UNIVERSE:
            idx = all_prices.index.get_loc(dt)
            ss  = sma_short.iloc[idx][t]
            sm  = sma_medium.iloc[idx][t]
            sl  = sma_long.iloc[idx][t]
            av  = atr_vals.iloc[idx][t]
            p   = price_row[t]

            mag = compute_composite_signal(p, ss, sm, sl, av,
                                           SIGNAL_WEIGHTS, SIGNAL_TEMPERATURE,
                                           ALPHA_MIN_MAGNITUDE)
            if mag is None:
                continue
            direction = "Up" if mag > 0 else "Down"
            cached_signals[t] = (direction, mag)

            log_signals.append({
                'date': dt.date(),
                'symbol': t,
                'direction': direction,
                'magnitude': round(mag, 6),
                'price': round(p, 4),
                'sma_short': round(float(ss), 4) if not pd.isna(ss) else '',
                'sma_medium': round(float(sm), 4) if not pd.isna(sm) else '',
                'sma_long': round(float(sl), 4) if not pd.isna(sl) else '',
                'atr': round(float(av), 4) if not pd.isna(av) else '',
            })
    else:
        alpha_trading_days += 1
        is_rebalance_day    = False

    # ── PCM: compute targets ──────────────────────────────────────────────────
    counter_rebalance = (
        trading_days_since_rebalance is None or
        trading_days_since_rebalance >= REBALANCE_INTERVAL
    )
    rebalance_today = is_rebalance_day or counter_rebalance

    flip_symbols = set()

    if rebalance_today:
        trading_days_since_rebalance = 1
        current_week_id = dt.strftime('%Y-%m-%d')

        # Snapshot actual weights
        actual_weights = {}
        for t in EQUITY_UNIVERSE:
            val = positions[t] * price_row[t]
            if abs(val) > 0:
                actual_weights[t] = val / nav if nav > 0 else 0

        # Compute raw weights from signals
        raw_weights = {}
        for t, (direction, mag) in cached_signals.items():
            sign = 1.0 if direction == "Up" else -1.0
            raw_weights[t] = sign * abs(mag)

        total_abs = sum(abs(w) for w in raw_weights.values())
        if total_abs > 1e-8:
            weights = {s: w / total_abs for s, w in raw_weights.items()}
            vol = estimate_vol(weights, rolling_returns)
            if vol and vol > 1e-8:
                scale = TARGET_VOL / vol
                weights = {s: w * scale for s, w in weights.items()}
            weights = apply_per_name_cap(weights, MAX_WEIGHT)
            weights = apply_gross_cap(weights, MAX_GROSS)
            weights = apply_net_cap(weights, MAX_NET)
            weekly_targets   = weights
            signal_strengths = {t: abs(mag) for t, (_, mag) in cached_signals.items() if t in weights}
        else:
            weekly_targets   = {}
            signal_strengths = {}

        # Classify
        all_syms = set(weekly_targets) | set(previous_weekly_targets) | set(actual_weights)
        classifications = {}
        for sym in all_syms:
            new_w    = weekly_targets.get(sym, 0.0)
            old_w    = previous_weekly_targets.get(sym, 0.0)
            actual_w = actual_weights.get(sym, 0.0)
            has_new  = sym in weekly_targets
            was_tgt  = sym in previous_weekly_targets
            is_held  = abs(actual_w) > 1e-8

            if has_new and not was_tgt and not is_held:
                classifications[sym] = "NEW_ENTRY"
            elif has_new and not was_tgt and is_held:
                classifications[sym] = "FLIP" if (actual_w > 0) != (new_w > 0) else "RESIZE"
            elif not has_new and (was_tgt or is_held):
                classifications[sym] = "EXIT"
            elif not has_new:
                continue
            elif (new_w > 0) != (old_w > 0):
                classifications[sym] = "FLIP"
            elif abs(new_w - actual_w) <= DEAD_BAND:
                classifications[sym] = "HOLD"
            else:
                classifications[sym] = "RESIZE"

        last_classifications = classifications

        # Update scale state
        for sym, action in classifications.items():
            if action == "NEW_ENTRY":
                symbol_scale_state[sym] = {"scale_day": 0, "is_scaling": True}
            elif action == "FLIP":
                symbol_scale_state[sym] = {"scale_day": 0, "is_scaling": True}
                flip_symbols.add(sym)
            elif action == "RESIZE":
                symbol_scale_state[sym] = {"scale_day": 0, "is_scaling": False}
            elif action == "HOLD":
                symbol_scale_state[sym] = {"scale_day": 0, "is_scaling": False}
            elif action == "EXIT":
                symbol_scale_state.pop(sym, None)

        previous_weekly_targets = dict(weekly_targets)

    else:
        trading_days_since_rebalance += 1
        for sym, state in symbol_scale_state.items():
            if state["is_scaling"]:
                if state["scale_day"] < SCALING_DAYS - 1:
                    state["scale_day"] += 1
                else:
                    state["is_scaling"] = False

    # ── Cancel stale limit orders (week_id based) ────────────────────────────
    still_open = []
    for order in open_limit_orders:
        if order['week_id'] < current_week_id:
            log_order_events.append({
                'date': dt.date(), 'order_id': order['order_id'],
                'symbol': order['symbol'], 'status': 'Canceled',
                'direction': order['direction'], 'quantity': order['quantity'],
                'fill_quantity': 0, 'fill_price': 0,
                'order_type': 'Limit', 'limit_price': order['limit_price'],
                'market_price_at_submit': order['market_price_at_submit'], 'tag': order['tag']
            })
        else:
            still_open.append(order)
    open_limit_orders = still_open

    # ── Try to fill open limit orders ────────────────────────────────────────
    still_open = []
    for order in open_limit_orders:
        t   = order['symbol']
        p   = price_row[t]
        qty = order['quantity']
        lp  = order['limit_price']
        filled = False

        if qty > 0 and p <= lp:   # buy limit fills if price drops to limit
            filled = True
        elif qty < 0 and p >= lp: # sell limit fills if price rises to limit
            filled = True

        if filled:
            fill_price = lp
            cash -= qty * fill_price
            positions[t] += qty
            slip_bps = (fill_price - order['market_price_at_submit']) / order['market_price_at_submit'] * 10_000
            log_slippage.append({
                'date': dt.date(), 'order_id': order['order_id'],
                'symbol': t, 'direction': order['direction'],
                'quantity': qty, 'expected_price': order['market_price_at_submit'],
                'fill_price': fill_price,
                'slippage_bps': round(slip_bps, 4)
            })
            log_order_events.append({
                'date': dt.date(), 'order_id': order['order_id'],
                'symbol': t, 'status': 'Filled',
                'direction': order['direction'], 'quantity': qty,
                'fill_quantity': qty, 'fill_price': fill_price,
                'order_type': 'Limit', 'limit_price': lp,
                'market_price_at_submit': order['market_price_at_submit'],
                'tag': order['tag']
            })
        else:
            still_open.append(order)
    open_limit_orders = still_open

    # ── Generate new order targets ────────────────────────────────────────────
    nav = get_nav(day_idx, price_row)

    # Exits (market orders)
    if rebalance_today:
        for sym, action in last_classifications.items():
            if action in ("EXIT", "FLIP"):
                qty_held = positions.get(sym, 0.0)
                if abs(qty_held) > 0.01:
                    p = price_row[sym]
                    cash += qty_held * p  # market fill
                    slip_bps = 0.0  # market order, no slippage modelled
                    oid = next_order_id()
                    direction = "Sell" if qty_held > 0 else "Buy"
                    log_slippage.append({
                        'date': dt.date(), 'order_id': oid,
                        'symbol': sym, 'direction': direction,
                        'quantity': -qty_held,
                        'expected_price': p, 'fill_price': p,
                        'slippage_bps': 0.0
                    })
                    log_order_events.append({
                        'date': dt.date(), 'order_id': oid, 'symbol': sym,
                        'status': 'Filled', 'direction': direction,
                        'quantity': -qty_held, 'fill_quantity': -qty_held,
                        'fill_price': p, 'order_type': 'Market',
                        'limit_price': None, 'market_price_at_submit': p,
                        'tag': f'exit|week:{current_week_id}'
                    })
                    positions[sym] = 0.0

    nav = get_nav(day_idx, price_row)

    # Entries / scaling (limit orders)
    for sym, final_w in weekly_targets.items():
        if sym in flip_symbols and rebalance_today:
            continue
        state = symbol_scale_state.get(sym, {"scale_day": 0, "is_scaling": False})
        if not state["is_scaling"] and not (rebalance_today and last_classifications.get(sym) == "RESIZE"):
            continue

        sig_str  = signal_strengths.get(sym, 0.5)
        schedule = get_schedule(sig_str)
        day_idx2 = min(state["scale_day"], len(schedule)-1)
        today_w  = final_w * schedule[day_idx2]

        target_val   = today_w * nav
        current_val  = positions[sym] * price_row[sym]
        delta_val    = target_val - current_val
        p            = price_row[sym]
        if p <= 0:
            continue
        qty = round(delta_val / p)
        if qty == 0:
            continue

        tier  = get_tier_label(sig_str)
        direction = "Buy" if qty > 0 else "Sell"
        if tier == "strong":
            offset = 0.0
        elif tier == "moderate":
            offset = MODERATE_OFFSET
        else:
            offset = WEAK_OFFSET

        # Limit price: buy below market, sell above market
        if qty > 0:
            limit_p = round(p * (1 - offset), 4)
        else:
            limit_p = round(p * (1 + offset), 4)

        oid = next_order_id()
        tag = f'{tier}|ss:{sig_str:.3f}|week:{current_week_id}|day:{state["scale_day"]}'
        open_limit_orders.append({
            'order_id': oid, 'symbol': sym, 'quantity': qty,
            'limit_price': limit_p, 'direction': direction,
            'market_price_at_submit': p, 'week_id': current_week_id,
            'tag': tag, 'tier': tier
        })
        log_order_events.append({
            'date': dt.date(), 'order_id': oid, 'symbol': sym,
            'status': 'Submitted', 'direction': direction,
            'quantity': qty, 'fill_quantity': 0, 'fill_price': 0,
            'order_type': 'Limit', 'limit_price': limit_p,
            'market_price_at_submit': p, 'tag': tag
        })

    # ── Log targets state ─────────────────────────────────────────────────────
    nav = get_nav(day_idx, price_row)
    for sym, final_w in weekly_targets.items():
        state = symbol_scale_state.get(sym, {"scale_day": 0, "is_scaling": False})
        sig_str  = signal_strengths.get(sym, 0.5)
        schedule = get_schedule(sig_str)
        day_idx2 = min(state["scale_day"], len(schedule)-1)
        sched_frac = schedule[day_idx2]
        actual_w = positions[sym] * price_row[sym] / nav if nav > 0 else 0

        log_targets.append({
            'date': dt.date(),
            'symbol': sym,
            'week_id': current_week_id,
            'scale_day': state["scale_day"],
            'is_scaling': state["is_scaling"],
            'weekly_target_w': round(final_w, 6),
            'scheduled_fraction': round(sched_frac, 6),
            'scheduled_w': round(final_w * sched_frac, 6),
            'actual_w': round(actual_w, 6),
            'classification': last_classifications.get(sym, ''),
            'signal_strength': round(signal_strengths.get(sym, 0), 6),
        })

    # ── Log daily snapshot ────────────────────────────────────────────────────
    nav = get_nav(day_idx, price_row)
    long_val  = sum(positions[t] * price_row[t] for t in EQUITY_UNIVERSE if positions[t] > 0)
    short_val = sum(abs(positions[t] * price_row[t]) for t in EQUITY_UNIVERSE if positions[t] < 0)
    gross_exp = (long_val + short_val) / nav if nav > 0 else 0
    net_exp   = (long_val - short_val) / nav if nav > 0 else 0
    num_pos   = sum(1 for t in EQUITY_UNIVERSE if abs(positions[t]) > 0.01)
    daily_pnl = nav - prev_nav if prev_nav else 0
    cum_pnl   = nav - starting_cash

    # Estimate portfolio vol
    w_dict = {t: positions[t] * price_row[t] / nav for t in EQUITY_UNIVERSE if abs(positions[t]) > 0.01 and nav > 0}
    est_vol = estimate_vol(w_dict, rolling_returns) if w_dict else None

    log_snapshots.append({
        'date': dt.date(),
        'nav': round(nav, 2),
        'cash': round(cash, 2),
        'gross_exposure': round(gross_exp, 4),
        'net_exposure': round(net_exp, 4),
        'long_exposure': round(long_val/nav if nav>0 else 0, 4),
        'short_exposure': round(short_val/nav if nav>0 else 0, 4),
        'daily_pnl': round(daily_pnl, 2),
        'cumulative_pnl': round(cum_pnl, 2),
        'daily_slippage': 0.0,
        'num_positions': num_pos,
        'estimated_vol': round(est_vol, 4) if est_vol else '',
    })

    # Log positions
    for t in EQUITY_UNIVERSE:
        qty = positions[t]
        if abs(qty) > 0.01:
            p   = price_row[t]
            val = qty * p
            log_positions.append({
                'date': dt.date(), 'symbol': t,
                'quantity': round(qty, 4),
                'price': round(p, 4),
                'market_value': round(val, 2),
                'weight': round(val/nav if nav>0 else 0, 4),
            })

    prev_nav = nav

    if day_idx % 50 == 0:
        print(f"  {dt.date()}  NAV=${nav:,.0f}  Positions={num_pos}  Gross={gross_exp:.1%}")

# ── Save CSVs ─────────────────────────────────────────────────────────────────
print("\nSaving CSVs...")

def save(records, filename):
    df = pd.DataFrame(records)
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"  Saved {filename}: {len(df):,} rows")
    return df

snap_df   = save(log_snapshots,    "daily_snapshots.csv")
pos_df    = save(log_positions,    "positions.csv")
sig_df    = save(log_signals,      "signals.csv")
slip_df   = save(log_slippage,     "slippage.csv")
tgt_df    = save(log_targets,      "targets.csv")
oe_df     = save(log_order_events, "order_events.csv")

# SPY benchmark (equal-weight index proxy)
spy_bt = bt_prices.mean(axis=1)
spy_df = spy_bt.reset_index()
spy_df.columns = ['date', 'price']
spy_df['date'] = spy_df['date'].dt.date
spy_df.to_csv(os.path.join(OUTPUT_DIR, "spy_benchmark.csv"), index=False)
print(f"  Saved spy_benchmark.csv: {len(spy_df)} rows")

print("\n✓ Simulation complete!")
print(f"  Final NAV : ${snap_df['nav'].iloc[-1]:,.2f}")
print(f"  Total Ret : {(snap_df['nav'].iloc[-1]/starting_cash - 1)*100:+.2f}%")
print(f"  Files in  : {OUTPUT_DIR}/")

Generating synthetic price data...
Price data: 793 total days | 523 backtest days
Computing indicators...
Running backtest simulation...
  2020-01-01  NAV=$100,000  Positions=0  Gross=0.0%
  2020-03-11  NAV=$97,788  Positions=2  Gross=0.2%
  2020-05-20  NAV=$97,787  Positions=10  Gross=1.9%
  2020-07-29  NAV=$98,953  Positions=7  Gross=2.9%
  2020-10-07  NAV=$99,856  Positions=7  Gross=3.8%
  2020-12-16  NAV=$98,872  Positions=12  Gross=8.1%
  2021-02-24  NAV=$98,734  Positions=8  Gross=6.2%
  2021-05-05  NAV=$97,981  Positions=5  Gross=2.6%
  2021-07-14  NAV=$96,617  Positions=2  Gross=1.7%
  2021-09-22  NAV=$95,413  Positions=16  Gross=6.3%
  2021-12-01  NAV=$94,860  Positions=13  Gross=6.5%

Saving CSVs...
  Saved daily_snapshots.csv: 523 rows
  Saved positions.csv: 4,991 rows
  Saved signals.csv: 3,110 rows
  Saved slippage.csv: 921 rows
  Saved targets.csv: 15,492 rows
  Saved order_events.csv: 3,631 rows
  Saved spy_benchmark.csv: 523 rows

✓ Simulation complete!
  Final NAV : $9